In [1]:
# Already installed
!pip3 install langchain langchain-community pypdf
!pip3 install sentence-transformers
!pip3 install chromadb


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

/Users/raghul/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading PDF Documents

In [3]:
def load_all_pdfs(folder_path):
 
    all_documents = []
 
    all_files = os.listdir(folder_path)
 
    all_files.sort()
 
    for filename in all_files:
 
        if not filename.lower().endswith(".pdf"):
            continue

        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}...")
 
        try:
            loader = PyPDFLoader(file_path)
            pages = loader.load()
 
            all_documents.extend(pages)
 
            print(f"Loaded {len(pages)} pages from {filename}")
 
        except Exception as e:
            print(f"Error loading {filename}: {e}")
 
    return all_documents

In [4]:
documents = load_all_pdfs("documents")
print(f"Total pages loaded: {len(documents)}")

Loading: 2759-7504-70-6-0400.pdf...
Loaded 8 pages from 2759-7504-70-6-0400.pdf
Loading: Alzheimer's extended.pdf...
Loaded 39 pages from Alzheimer's extended.pdf
Loading: Alzheimer's.pdf...
Loaded 31 pages from Alzheimer's.pdf
Loading: diabetes.pdf...
Loaded 20 pages from diabetes.pdf
Loading: medscimonit-30-e945091.pdf...
Loaded 7 pages from medscimonit-30-e945091.pdf
Total pages loaded: 105


In [5]:
for i, doc in enumerate(documents[:3]):
    print(f"Source: {doc.metadata['source']}, Page: {doc.metadata['page']}")
    print(doc.page_content[:300])
    print("---")

Source: documents/2759-7504-70-6-0400.pdf, Page: 0
400
Nishida Y, Watada H: The up-to-date for diabetes
Introduction
Diabetes mellitus is a chronic disease that continues 
to increase worldwide and requires proper manage-
ment and treatment. This review includes a general 
description of diabetes, its history, the basics of 
treatment, the latest tr
---
Source: documents/2759-7504-70-6-0400.pdf, Page: 1
401
Juntendo Medical Journal 70(6), 2024
levels. It is the only hormone that lowers blood 
glucose levels and is essential for incorporating 
glucose into the cells, using it as energy, or storing 
the excess glucose as lipids. 
Types of diabetes
There are two main types of diabetes: type 1 
diabete
---
Source: documents/2759-7504-70-6-0400.pdf, Page: 2
402
Nishida Y, Watada H: The up-to-date for diabetes
less to prevent complications 3).
Symptoms in diabetes
High blood glucose provokes symptoms of diabetes, 
such as thirst, polydipsia, polyuria, and weight loss. 
However, these symptom

## Chunking Documents

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)
 
print(f"\nBefore splitting: {len(documents)} pages")
print(f"After splitting:  {len(chunks)} chunks")
print(f"Average chunks per page: {len(chunks) / len(documents):.1f}")


Before splitting: 105 pages
After splitting:  839 chunks
Average chunks per page: 8.0


In [7]:
print("SAMPLE CHUNKS")
 
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(f"Source: {chunk.metadata['source']}")
    print(f"Page:   {chunk.metadata['page']}")
    print(f"Length: {len(chunk.page_content)} characters")
    print(f"Text preview:")
    print(f"  {chunk.page_content[:200]}...")
    print()

SAMPLE CHUNKS

--- Chunk 1 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 981 characters
Text preview:
  400
Nishida Y, Watada H: The up-to-date for diabetes
Introduction
Diabetes mellitus is a chronic disease that continues 
to increase worldwide and requires proper manage-
ment and treatment. This revi...


--- Chunk 2 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 985 characters
Text preview:
  he suffered from various complications related to 
diabetes, such as thirst, blindness, and chest pain.
Basics of diabetes mellitus
What is diabetes?
Diabetes mellitus is a condition in which blood 
g...


--- Chunk 3 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 886 characters
Text preview:
  management and treatment advancements. Effective diabetes management includes maintaining blood glucose levels within 
normal ranges and monitoring HbA1c, a marker reflecting average glucose levels ov...



## Generating Embeddings for Chunks

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np 
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
print("Model loaded!")


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11901.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [9]:
chunk_texts = [chunk.page_content for chunk in chunks]
 
print(f"Embedding {len(chunk_texts)} chunks...")
print("(This may take 30-60 seconds depending on your machine)")

all_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64
)

print(f"\n Embedding complete!")
print(f"  Chunks embedded: {len(all_embeddings)}")
print(f"  Embedding shape: {all_embeddings.shape}")
print(f"  Total memory: {all_embeddings.nbytes / 1024 / 1024:.1f} MB")

Embedding 839 chunks...
(This may take 30-60 seconds depending on your machine)

 Embedding complete!
  Chunks embedded: 839
  Embedding shape: (839, 384)
  Total memory: 1.2 MB


## Storing Embeddings in ChromaDB

In [10]:
import chromadb

CHROMA_PATH = "chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="medical_docs",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB loaded! Documents in collection: {collection.count()}")

ChromaDB loaded! Documents in collection: 839


### Adding the chunks to the collection

In [11]:
if collection.count() == 0:
    print(f"Adding {len(chunks)} chunks with pre-computed embeddings...")

    documents_list = [chunk.page_content for chunk in chunks]
    metadatas_list = [chunk.metadata for chunk in chunks]
    ids_list = [f"doc_{i}" for i in range(len(chunks))]

    BATCH_SIZE = 500

    for start in range(0, len(chunks), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(chunks))

        collection.add(
            documents=documents_list[start:end],
            metadatas=metadatas_list[start:end],
            ids=ids_list[start:end],
            embeddings=all_embeddings[start:end].tolist()  # our Step 4 embeddings
        )

        print(f"  Batch {start // BATCH_SIZE + 1}: added chunks {start} to {end - 1}")

    print(f"\n All {len(chunks)} chunks stored with pre-computed embeddings!")
else:
    print(f"Collection already has {collection.count()} documents. Skipping.")

Collection already has 839 documents. Skipping.


In [12]:
sample = collection.peek(limit=3)
 
for i in range(min(3, len(sample['ids']))):
    print(f"\n--- Stored Document: {sample['ids'][i]} ---")
    print(f"  Source: {sample['metadatas'][i].get('source', 'N/A')}")
    print(f"  Page:   {sample['metadatas'][i].get('page', 'N/A')}")
    print(f"  Text:   {sample['documents'][i][:150]}...")
    print(f"  Embeddings:   {sample['embeddings'][i][:150]}...")


--- Stored Document: doc_0 ---
  Source: documents/2759-7504-70-6-0400.pdf
  Page:   0
  Text:   400
Nishida Y, Watada H: The up-to-date for diabetes
Introduction
Diabetes mellitus is a chronic disease that continues 
to increase worldwide and req...
  Embeddings:   [-6.28945604e-02  1.20206527e-01 -5.16719408e-02  7.40096420e-02
 -1.03094004e-01 -4.95618396e-03  9.78769362e-02  8.93808901e-02
 -6.85888380e-02 -3.89334597e-02  1.30577534e-02 -1.07889865e-02
 -2.91038696e-02  8.79827887e-02 -4.29274887e-02 -7.68452808e-02
 -5.54292500e-02  2.27595512e-02  1.50779514e-02  3.66584361e-02
  2.63194125e-02  5.74750826e-02  8.07511881e-02 -4.45137918e-02
  2.54371949e-02 -2.27573868e-02  7.52966329e-02 -4.87493053e-02
 -7.21444283e-03  3.33865080e-03 -8.01958218e-02 -3.40195303e-03
  8.87239501e-02  1.15462029e-02  1.62561296e-03  5.26660681e-02
  4.92149889e-02  1.06788864e-02 -9.16167498e-02 -4.63692099e-02
  3.76530886e-02  3.52947488e-02  2.37850510e-02  2.19027102e-02
  3.73856537e-02 

## Retrieval Function

    Args:
        query: The user's question (a plain string)
        n_results: How many results to return (default 5)
 
    Returns:
        A list of dictionaries, each containing:
          - text: the chunk content
          - source: which PDF it came from
          - page: which page number
          - similarity: cosine similarity score (0 to 1)

In [13]:
def search_documents(query, n_results=5):
    raw_results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
 
    # Reformat into a cleaner structure
    results = []
 
    for i in range(len(raw_results['ids'][0])):
        distance = raw_results['distances'][0][i]
        similarity = 1 - distance
 
        results.append({
            "text": raw_results['documents'][0][i],
            "source": raw_results['metadatas'][0][i]['source'],
            "page": raw_results['metadatas'][0][i]['page'],
            "similarity": round(similarity, 4)
        })
 
    return results
 
 
def display_results(query, results):
    print(f"\nQuery: \"{query}\"")
 
    for i, r in enumerate(results):
        print(f"\n--- Result {i + 1} (similarity: {r['similarity']:.4f}) ---")
        print(f"Source: {r['source']}")
        print(f"Page:   {r['page']}")
        print(f"Text:   {r['text'][:300]}...")
 
    print()

## In-Scope Queries

In [14]:
query1 = "What are the symptoms of diabetes?"
results1 = search_documents(query1)
display_results(query1, results1)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11435.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: "What are the symptoms of diabetes?"

--- Result 1 (similarity: 0.7329) ---
Source: documents/diabetes.pdf
Page:   1
Text:   represents just a small portion of the enormous number of
unidentiﬁed cases. It is a chronic illness stemming from either
inadequate synthesis of insulin by the pancreas or inef ﬁcient
utilization of that production by the body. Uncontrolled diabetes
frequently results in hyperglycemia, or elevated ...

--- Result 2 (similarity: 0.5858) ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   2
Text:   402
Nishida Y, Watada H: The up-to-date for diabetes
less to prevent complications 3).
Symptoms in diabetes
High blood glucose provokes symptoms of diabetes, 
such as thirst, polydipsia, polyuria, and weight loss. 
However, these symptoms often go unnoticed until 
the disease progresses, so regular ...

--- Result 3 (similarity: 0.5365) ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Text:   he suffered from various complications related to 
diabetes, 

In [15]:
query2 = "What are the risk factors for Alzheimer's disease?"
results2 = search_documents(query2)
display_results(query2, results2)


Query: "What are the risk factors for Alzheimer's disease?"

--- Result 1 (similarity: 0.6835) ---
Source: documents/Alzheimer's extended.pdf
Page:   4
Text:   cases linked to AD, depending on the population and whether
biomarkers were used alongside clinical diagnosis (
18).
/four.tnum Genetics of Alzheimer’s disease
Despite the overall sporadic nature of the disease, many
studies have shown that Genetic mutations and gene variance have
been linked to an ...

--- Result 2 (similarity: 0.6668) ---
Source: documents/Alzheimer's extended.pdf
Page:   37
Text:   Geriatr Psychiatry Neurol. (2006) 19:78–82. doi: 10.1177/0891988706286505
524. Wood WG, Igbavboa U, Eckert GP , Johnson-Anuna LN, Müller WE. Is
hypercholesterolemia a risk factor for Alzheimer’s disease? Mol Neurobiol. (2005)
31:185–92. doi: 10.1385/MN:31:1-3:185
525. Litke R, Garcharna LC, Jiwani S...

--- Result 3 (similarity: 0.6641) ---
Source: documents/medscimonit-30-e945091.pdf
Page:   4
Text:   ness of dementia; hypertensi

## Cross-topic Query

In [16]:
query3 = "Is there a link between diabetes and Alzheimer's disease?"
results3 = search_documents(query3)
display_results(query3, results3)


Query: "Is there a link between diabetes and Alzheimer's disease?"

--- Result 1 (similarity: 0.7142) ---
Source: documents/Alzheimer's extended.pdf
Page:   29
Text:   AD Study. J Alzheimers Dis. (2022) 85:235–47. doi: 10.3233/JAD-215153
168. Marseglia A, Fratiglioni L, Kalpouzos G, Wang R, Bäckman L, Xu W.
Prediabetes and diabetes accelerate cognitive decline and pre dict microvascular
lesions: a population-based cohort study. Alzheimers Dement. (2019) 15:25–
33....

--- Result 2 (similarity: 0.6984) ---
Source: documents/Alzheimer's extended.pdf
Page:   37
Text:   Geriatr Psychiatry Neurol. (2006) 19:78–82. doi: 10.1177/0891988706286505
524. Wood WG, Igbavboa U, Eckert GP , Johnson-Anuna LN, Müller WE. Is
hypercholesterolemia a risk factor for Alzheimer’s disease? Mol Neurobiol. (2005)
31:185–92. doi: 10.1385/MN:31:1-3:185
525. Litke R, Garcharna LC, Jiwani S...

--- Result 3 (similarity: 0.6628) ---
Source: documents/Alzheimer's extended.pdf
Page:   8
Text:   with diabetes mellitus

## Out of Scope Query

In [17]:
query4 = "How do black holes form in space?"
results4 = search_documents(query4)
display_results(query4, results4)


Query: "How do black holes form in space?"

--- Result 1 (similarity: 0.1407) ---
Source: documents/Alzheimer's extended.pdf
Page:   6
Text:   pathological processes arising from A β accumulation, there may
also be a bidirectional link between sleep-wake disruption and tau
pathology (
68). Lucey et al. found that the non-rapid eye movement
(NREM) slow wave activity was inversely linked with tauopathy,
suggesting that changes in NREM slow w...

--- Result 2 (similarity: 0.1396) ---
Source: documents/diabetes.pdf
Page:   17
Text:   therapeutic agents for management of diabetes mellitus: A hope for drug designing
against diabetes mellitus. Life. (2024) 14:99. doi: 10.3390/life14010099
53. Silver B, Ramaiya K, Andrew SB, Fredrick O, Bajaj S, Kalra S, et al. ADSG
guidelines: insulin therapy in diabetes. Diabetes Ther . (2018) 9:4...

--- Result 3 (similarity: 0.1389) ---
Source: documents/diabetes.pdf
Page:   10
Text:   increasing the mass and function of pancreatic b-cells; increasing
ins

### Inference
- **In-scope queries** (diabetes, Alzheimer's) return similarity scores **0.58–0.73**, pulling from the correct PDFs.
- **Cross-topic query** (diabetes–Alzheimer's link) retrieves chunks from **both** topics — embeddings capture semantic connections across papers.
- **Out-of-scope query** (black holes) scores significantly **lower (~0.2–0.3)**, confirming the model distinguishes relevant from irrelevant content.

## Connecting Local LLM (Ollama + Llama 3)

In [18]:
import requests
import json
 
OLLAMA_URL = "http://localhost:11434"
 
try:
    response = requests.get(f"{OLLAMA_URL}/api/tags")
    models = [m['name'] for m in response.json()['models']]
    print(f" Ollama is running! Available models: {models}")
except requests.ConnectionError:
    print(" Ollama is not running!")
    print(" Open a terminal and run: ollama serve")

 Ollama is running! Available models: ['llama3:latest']


## Building the prompt template

In [19]:
def build_prompt(query, retrieved_chunks):
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks):
        source = chunk['source'].replace('documents/', '')
        page = chunk['page']
        context_parts.append(
            f"[Source: {source}, Page {page + 1}]\n{chunk['text']}"
        )
 
    context = "\n\n---\n\n".join(context_parts)
 
    prompt = f"""You are a medical research assistant. Answer the question based ONLY on the provided context from research papers. If the context does not contain enough information to answer the question, say "I don't have enough information in my documents to answer this question."
 
Keep your answer concise (3-5 sentences). Always mention which source the information comes from.
 
CONTEXT:
{context}
 
QUESTION: {query}
 
ANSWER:"""
 
    return prompt

## Ollama API call

In [20]:
def ask_llm(prompt, model="llama3"):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,      
            "options": {
                "temperature": 0.3,  
                "num_predict": 512   
            }
        }
    )
 
    if response.status_code == 200:
        return response.json()['response']
    else:
        return f"Error: {response.status_code} - {response.text}"

## Full RAG pipeline

In [21]:
def ask_rag(query, n_results=5):
 
    # Retrieve
    retrieved = search_documents(query, n_results=n_results)
 
    # Build prompt
    prompt = build_prompt(query, retrieved)
 
    # Generate answer
    answer = ask_llm(prompt)
 
    return {
        "query": query,
        "answer": answer,
        "sources": retrieved
    }

## Testing With Questions

In [22]:
questions = [
    "What treatments are available for Alzheimer's disease?",
    "How is type 2 diabetes diagnosed?",
    "Is there a connection between diabetes and Alzheimer's?",
]
 
for q in questions:
    result = ask_rag(q)
    print(f"\nQ: {result['query']}")
    print(f"A: {result['answer']}")
    print("-" * 60)


Q: What treatments are available for Alzheimer's disease?
A: According to the provided context, treatments for Alzheimer's disease aim to improve symptoms of cognitive and behavioral changes [12]. The main approaches include cholinesterase inhibitors and NMDA receptor antagonists, which provide symptomatic relief but do not slow disease progression [Alzheimer's extended.pdf, Page 2]. Additionally, emerging therapies such as amyloid-beta and tau-targeting treatments, gene therapy, and immunotherapy offer potential for disease modification [Alzheimer's extended.pdf, Page 2].

Source: Alzheimer's extended.pdf, Page 2
------------------------------------------------------------

Q: How is type 2 diabetes diagnosed?
A: I don't have enough information in my documents to answer this question. The provided context only discusses the basics of diabetes mellitus, types of diabetes, and symptoms, but does not mention the diagnostic process for type 2 diabetes specifically.
----------------------

In [23]:
result = ask_rag("What are the symptoms of diabetes?")
print(result['answer'])

According to the provided context from research papers, some of the early signs and/or symptoms of hyperglycemia include:

* Increased thirst (polydipsia) and/or hunger
* Frequent urination
* Headache
* Blurred vision
* Fatigue
* Weight loss
* Vaginal yeast infection
* Skin infection
* Slow healing of wounds

These symptoms are mentioned in [Source: diabetes.pdf, Page 2] as early signs and/or symptoms of hyperglycemia. Additionally, high blood glucose can lead to complications such as neuropathy (numbness), retinopathy (vision loss), and nephropathy (kidney failure) [Source: 2759-7504-70-6-0400.pdf, Page 3].


### Debugging: query phrasing affects retrieval
Rephrasing "How is type 2 diabetes diagnosed?" → "diagnosis criteria HbA1c blood glucose test" improved the top similarity score from 0.61 to 0.71 and retrieved the correct ADA diagnostic guidelines chunk.

In [24]:
results = search_documents("How is type 2 diabetes diagnosed?")
for i, r in enumerate(results):
    source = r['source'].replace('documents/', '')
    print(f"\nResult {i+1} (sim: {r['similarity']:.4f}) — {source}, Page {r['page'] + 1}")
    print(r['text'][:200])


Result 1 (sim: 0.6139) — 2759-7504-70-6-0400.pdf, Page 2
401
Juntendo Medical Journal 70(6), 2024
levels. It is the only hormone that lowers blood 
glucose levels and is essential for incorporating 
glucose into the cells, using it as energy, or storing 
th

Result 2 (sim: 0.5967) — 2759-7504-70-6-0400.pdf, Page 1
he suffered from various complications related to 
diabetes, such as thirst, blindness, and chest pain.
Basics of diabetes mellitus
What is diabetes?
Diabetes mellitus is a condition in which blood 
g

Result 3 (sim: 0.5955) — 2759-7504-70-6-0400.pdf, Page 3
402
Nishida Y, Watada H: The up-to-date for diabetes
less to prevent complications 3).
Symptoms in diabetes
High blood glucose provokes symptoms of diabetes, 
such as thirst, polydipsia, polyuria, and

Result 4 (sim: 0.5871) — diabetes.pdf, Page 18
20. Hurtado MD, Vella A. What is type 2 diabetes? Medicine. (2019) 47:10 –5.
doi: 10.1016/j.mpmed.2018.10.010
21. Dludla PV, Mabhida SE, Ziqubu K, Nkambule BB, Mazibuko-Mbeje 

In [25]:
results2 = search_documents("diagnosis criteria HbA1c blood glucose test")
for i, r in enumerate(results2):
    source = r['source'].replace('documents/', '')
    print(f"\nResult {i+1} (sim: {r['similarity']:.4f}) — {source}, Page {r['page'] + 1}")
    print(r['text'][:200])


Result 1 (sim: 0.7110) — diabetes.pdf, Page 8
TABLE 1 The ADA diagnostic guidelines: adapted from ( 6, 41).
Stages Latent Impaired
Glucose
Tolerance
(IGT)
Diabetes
Criteria
for
Diagnosis
If two or more
autoantibodies
are present
and the
glucose l

Result 2 (sim: 0.6940) — 2759-7504-70-6-0400.pdf, Page 2
diabetes is greatly influenced by genetic factors 
and lifestyle habits, such as diet and lack of exer -
cise (Figure 2)2).
Blood glucose management
Blood glucose control is very important in the 
tre

Result 3 (sim: 0.6182) — diabetes.pdf, Page 7
1.7 Diagnosis of DM
Generally, according to the American Diabetes Association
(ADA) Diagnostic Guidelines ( Table 1 ), there are four common
tests utilized for diagnosis of DM, namely: 1) Random (casu

Result 4 (sim: 0.6013) — 2759-7504-70-6-0400.pdf, Page 2
cations. A healthy person’s HbA1c is about 6% or 
less, while diabetic patients generally aim for 7% or 
Figure 1　A commemorative stamp for the 15th International 
Diabetes Federation Con

## Adding Source citations to every answers

#### Full RAG pipeline that returns a formatted answer with source citations and similarity scores.

In [26]:
def ask_rag_with_citations(query, n_results=5):
    retrieved = search_documents(query, n_results=n_results)
 
    prompt = build_prompt(query, retrieved)
    answer = ask_llm(prompt)
 
    citations = []
    seen = set() 
 
    for r in retrieved:
        source = r['source'].replace('documents/', '')
        page = r['page'] + 1  
        score = r['similarity']
        key = f"{source}_p{page}"
 
        if key not in seen:
            seen.add(key)
            citations.append({
                "source": source,
                "page": page,
                "similarity": score
            })
 
    return {
        "query": query,
        "answer": answer,
        "citations": citations
    }


def print_answer(result): 
    print(f"Q: {result['query']}\n")
    print(f"A: {result['answer']}\n")
    print("Sources:")
    print("-" * 50)
 
    for i, c in enumerate(result['citations']):
        print(f"  [{i+1}] {c['source']}")
        print(f"      Page {c['page']}  |  Score: {c['similarity']:.4f} ")
 
    print()

    

In [27]:
result = ask_rag_with_citations("What are the symptoms of diabetes?")
print_answer(result)

Q: What are the symptoms of diabetes?

A: According to the provided context from research papers, some of the early signs and/or symptoms of hyperglycemia (diabetes) include:

* Increased thirst (polydipsia)
* Frequent urination
* Headache
* Blurred vision
* Fatigue
* Weight loss
* Vaginal yeast infection
* Skin infection
* Slow healing of wounds

These symptoms are mentioned in [Source: diabetes.pdf, Page 2] and [Source: 2759-7504-70-6-0400.pdf, Page 3].

Additionally, prolonged high blood glucose can lead to complications such as neuropathy (numbness), retinopathy (vision loss), and nephropathy (kidney failure) [Source: 2759-7504-70-6-0400.pdf, Page 3].

Sources:
--------------------------------------------------
  [1] diabetes.pdf
      Page 2  |  Score: 0.7329 
  [2] 2759-7504-70-6-0400.pdf
      Page 3  |  Score: 0.5858 
  [3] 2759-7504-70-6-0400.pdf
      Page 1  |  Score: 0.5365 
  [4] diabetes.pdf
      Page 3  |  Score: 0.5195 
  [5] diabetes.pdf
      Page 7  |  Score: 0.5110

In [28]:
result2 = ask_rag_with_citations("What treatments are available for Alzheimer's disease?")
print_answer(result2)

Q: What treatments are available for Alzheimer's disease?

A: According to the provided context, treatments for Alzheimer's disease aim to improve symptoms of cognitive and behavioral changes [12]. The most common form, late-onset Alzheimer's disease, is characterized by extracellular amyloid deposits [13]. Treatments modify symptoms, but there is no cure. Patients with Alzheimer's disease have reduced cerebral levels of choline acetyltransferase [Source: medscimonit-30-e945091.pdf, Page 3].

Additionally, the sources mention various treatments and therapies being explored or already available, including:

* Cholinesterase inhibitors and NMDA receptor antagonists for symptomatic relief [Alzheimer's extended.pdf, Page 2]
* Disease-modifying therapies, such as amyloid-beta and tau-targeting treatments, gene therapy, and immunotherapy [Alzheimer's extended.pdf, Page 2]
* Nonpharmacological treatments like behavior modification, exercise, and cognitive training [Source: Alzheimer's.pdf, Pa

In [29]:
result3 = ask_rag_with_citations("Is there a connection between diabetes and Alzheimer's?")
print_answer(result3)

Q: Is there a connection between diabetes and Alzheimer's?

A: According to the provided context from research papers, there is a connection between diabetes and Alzheimer's disease. Studies have shown that individuals with diabetes or even prediabetic individuals are faced with an increased risk of cognitive deterioration, dementia, and Alzheimer's disease (167, 168). Additionally, long-term exposure to poorly controlled hyperglycemia has been correlated with a higher risk of developing dementia (170), and type 1 DM is associated with brain atrophy in the thalamus, putamen, and superior frontal gyrus (171).

Source: Alzheimer's extended.pdf, Page 9

Sources:
--------------------------------------------------
  [1] Alzheimer's extended.pdf
      Page 30  |  Score: 0.7119 
  [2] Alzheimer's extended.pdf
      Page 38  |  Score: 0.6780 
  [3] Alzheimer's extended.pdf
      Page 9  |  Score: 0.6589 



##  Hallucination guard
###     Complete RAG pipeline with hallucination guard. If the best retrieval score is below the threshold, refuses to answer instead of risking a hallucinated response.

In [30]:
SIMILARITY_THRESHOLD = 0.6
 
def ask_rag_safe(query, n_results=5, threshold=SIMILARITY_THRESHOLD):
    
    retrieved = search_documents(query, n_results=n_results)
 
    best_score = retrieved[0]['similarity'] if retrieved else 0
 
    if best_score < threshold:
        return {
            "query": query,
            "answer": (
                "I don't have enough information in my documents to answer "
                "this question reliably. The most relevant content I found "
                f"had a confidence score of {best_score:.2f}, which is below "
                f"the threshold of {threshold}."
            ),
            "citations": [],
            "confident": False,
            "best_score": best_score
        }
 
    prompt = build_prompt(query, retrieved)
    answer = ask_llm(prompt)
 
    citations = []
    seen = set()
    for r in retrieved:
        source = r['source'].replace('documents/', '')
        page = r['page'] + 1
        key = f"{source}_p{page}"
        if key not in seen:
            seen.add(key)
            citations.append({
                "source": source,
                "page": page,
                "similarity": r['similarity']
            })
 
    return {
        "query": query,
        "answer": answer,
        "citations": citations,
        "confident": True,
        "best_score": best_score
    }

def print_safe_answer(result):
 
    status = "PASS" if result['confident'] else "BLOCKED"
    print(f"[{status}] (best score: {result['best_score']:.4f})\n")
    print(f"Q: {result['query']}\n")
    print(f"A: {result['answer']}\n")
 
    if result['citations']:
        print("Sources:")
        print("-" * 50)
        for i, c in enumerate(result['citations']):
            print(f"  [{i+1}] {c['source']}")
            print(f"      Page {c['page']}  |  Score: {c['similarity']:.4f}")
    print()
    

## in-scope question (should PASS)

In [31]:
r1 = ask_rag_safe("What are the symptoms of diabetes?")
print_safe_answer(r1)

[PASS] (best score: 0.7329)

Q: What are the symptoms of diabetes?

A: According to the provided context from research papers, some of the early signs and/or symptoms of hyperglycemia (diabetes) include:

* Increased thirst (polydipsia)
* Frequent urination
* Headache
* Blurred vision
* Fatigue
* Weight loss
* Vaginal yeast infection
* Skin infection
* Slow healing of wounds

These symptoms are mentioned in [Source: diabetes.pdf, Page 2] and [Source: 2759-7504-70-6-0400.pdf, Page 3].

Additionally, high blood glucose can lead to complications such as neuropathy (numbness), retinopathy (vision loss), and nephropathy (kidney failure) over time.

Sources:
--------------------------------------------------
  [1] diabetes.pdf
      Page 2  |  Score: 0.7329
  [2] 2759-7504-70-6-0400.pdf
      Page 3  |  Score: 0.5858
  [3] 2759-7504-70-6-0400.pdf
      Page 1  |  Score: 0.5365
  [4] diabetes.pdf
      Page 3  |  Score: 0.5195
  [5] diabetes.pdf
      Page 7  |  Score: 0.5110



## out-of-scope question (should be BLOCKED)

In [33]:
r2 = ask_rag_safe("How do black holes form in space?")
print_safe_answer(r2)

[BLOCKED] (best score: 0.1407)

Q: How do black holes form in space?

A: I don't have enough information in my documents to answer this question reliably. The most relevant content I found had a confidence score of 0.14, which is below the threshold of 0.6.




## borderline question

In [34]:
r3 = ask_rag_safe("What is the role of exercise in managing diabetes?")
print_safe_answer(r3)

[PASS] (best score: 0.6821)

Q: What is the role of exercise in managing diabetes?

A: According to the provided context from research papers [Source: 2759-7504-70-6-0400.pdf, Page 4], exercise therapy is one of the basic pillars of diabetes treatment. Exercise improves the effectiveness of insulin and lowers blood glucose levels. A combination of aerobic exercise (aerobics) and resistance exercise (strength training) is considered effective [Source: 2759-7504-70-6-0400.pdf, Page 3]. Additionally, a study conducted by Juntendo University confirmed that when patients with type 2 diabetes in hospital adopt exercise therapy, the fat stored in muscles is reduced, and insulin efficacy is improved [Source: 2759-7504-70-6-0400.pdf, Page 4].

Sources:
--------------------------------------------------
  [1] 2759-7504-70-6-0400.pdf
      Page 4  |  Score: 0.6821
  [2] 2759-7504-70-6-0400.pdf
      Page 3  |  Score: 0.6770
  [3] diabetes.pdf
      Page 18  |  Score: 0.6296
  [4] diabetes.pdf
   

In [36]:
test_cases = [
    {
        "question": "What are the symptoms of diabetes?",
        "expected_source": "diabetes",
        "expected_keywords": ["thirst", "polyuria", "weight loss"],
        "in_scope": True
    },
    {
        "question": "What is type 1 diabetes?",
        "expected_source": "diabetes",
        "expected_keywords": ["autoimmune", "insulin"],
        "in_scope": True
    },
    {
        "question": "What treatments are available for Alzheimer's disease?",
        "expected_source": "alzheimer",
        "expected_keywords": ["cholinesterase", "inhibitor"],
        "in_scope": True
    },
    {
        "question": "What are the risk factors for Alzheimer's disease?",
        "expected_source": "alzheimer",
        "expected_keywords": ["age", "genetic"],
        "in_scope": True
    },
    {
        "question": "Is there a connection between diabetes and Alzheimer's?",
        "expected_source": "alzheimer",
        "expected_keywords": ["diabetes", "risk", "cognitive"],
        "in_scope": True
    },
    {
        "question": "What is HbA1c and why is it important?",
        "expected_source": "diabetes",
        "expected_keywords": ["glucose", "blood"],
        "in_scope": True
    },
    {
        "question": "What role does insulin play in the body?",
        "expected_source": "diabetes",
        "expected_keywords": ["insulin", "glucose"],
        "in_scope": True
    },
    {
        "question": "What is amyloid beta in Alzheimer's disease?",
        "expected_source": "alzheimer",
        "expected_keywords": ["amyloid", "plaque"],
        "in_scope": True
    },
    {
        "question": "How do black holes form in space?",
        "expected_source": None,
        "expected_keywords": [],
        "in_scope": False
    },
    {
        "question": "What is the recipe for chocolate cake?",
        "expected_source": None,
        "expected_keywords": [],
        "in_scope": False
    },
]
 
print(f"Defined {len(test_cases)} test cases")
print(f"  In-scope:     {sum(1 for t in test_cases if t['in_scope'])}")
print(f"  Out-of-scope: {sum(1 for t in test_cases if not t['in_scope'])}")

Defined 10 test cases
  In-scope:     8
  Out-of-scope: 2


In [41]:
import time
 
eval_results = []
 
print("Running evaluation...\n")
 
for i, test in enumerate(test_cases):
    start = time.time()
    result = ask_rag_safe(test['question'])
    elapsed = time.time() - start
 
    # Did the hallucination guard work correctly?
    if test['in_scope']:
        guard_correct = result['confident'] 
    else:
        guard_correct = not result['confident'] 
 
    # Did retrieval find the right source?
    retrieval_correct = False
    if result['citations'] and test['expected_source']:
        for c in result['citations']:
            if test['expected_source'].lower() in c['source'].lower():
                retrieval_correct = True
                break
    elif not test['in_scope'] and not result['citations']:
        retrieval_correct = True 
 
    # Does the answer contain expected keywords?
    answer_lower = result['answer'].lower()
    keywords_found = [
        kw for kw in test['expected_keywords']
        if kw.lower() in answer_lower
    ]
    keyword_score = (
        len(keywords_found) / len(test['expected_keywords'])
        if test['expected_keywords'] else 1.0
    )
 
    eval_results.append({
        "question": test['question'],
        "in_scope": test['in_scope'],
        "guard_correct": guard_correct,
        "retrieval_correct": retrieval_correct,
        "keyword_score": keyword_score,
        "keywords_found": keywords_found,
        "keywords_expected": test['expected_keywords'],
        "best_score": result['best_score'],
        "time": elapsed
    })
 
    status = "✓" if (guard_correct and retrieval_correct) else "✗"
    print(f"  {status} Q{i+1}: {test['question'][:50]}... ({elapsed:.1f}s)")
 
print("\nEvaluation complete!")


Running evaluation...

  ✓ Q1: What are the symptoms of diabetes?... (19.1s)
  ✓ Q2: What is type 1 diabetes?... (8.6s)
  ✓ Q3: What treatments are available for Alzheimer's dise... (11.5s)
  ✓ Q4: What are the risk factors for Alzheimer's disease?... (21.3s)
  ✓ Q5: Is there a connection between diabetes and Alzheim... (13.1s)
  ✗ Q6: What is HbA1c and why is it important?... (0.0s)
  ✗ Q7: What role does insulin play in the body?... (0.0s)
  ✓ Q8: What is amyloid beta in Alzheimer's disease?... (10.7s)
  ✓ Q9: How do black holes form in space?... (0.0s)
  ✓ Q10: What is the recipe for chocolate cake?... (0.0s)

Evaluation complete!


In [40]:
print("\nDETAILED RESULTS")
 
for i, r in enumerate(eval_results):
    print(f"\nQ{i+1}: {r['question']}")
    print(f"  In-scope:        {r['in_scope']}")
    print(f"  Guard correct:   {r['guard_correct']}")
    print(f"  Retrieval correct: {r['retrieval_correct']}")
    print(f"  Keyword score:   {r['keyword_score']:.0%} ({r['keywords_found']} / {r['keywords_expected']})")
    print(f"  Best sim score:  {r['best_score']:.4f}")
    print(f"  Response time:   {r['time']:.1f}s")


DETAILED RESULTS

Q1: What are the symptoms of diabetes?
  In-scope:        True
  Guard correct:   True
  Retrieval correct: True
  Keyword score:   67% (['thirst', 'weight loss'] / ['thirst', 'polyuria', 'weight loss'])
  Best sim score:  0.7329
  Response time:   14.5s

Q2: What is type 1 diabetes?
  In-scope:        True
  Guard correct:   True
  Retrieval correct: True
  Keyword score:   100% (['autoimmune', 'insulin'] / ['autoimmune', 'insulin'])
  Best sim score:  0.6454
  Response time:   8.2s

Q3: What treatments are available for Alzheimer's disease?
  In-scope:        True
  Guard correct:   True
  Retrieval correct: True
  Keyword score:   100% (['cholinesterase', 'inhibitor'] / ['cholinesterase', 'inhibitor'])
  Best sim score:  0.7254
  Response time:   16.1s

Q4: What are the risk factors for Alzheimer's disease?
  In-scope:        True
  Guard correct:   True
  Retrieval correct: True
  Keyword score:   100% (['age', 'genetic'] / ['age', 'genetic'])
  Best sim score:  